# From capsule to eKG

This notebook has the minimum sample code to convert a series of 'capsules' representing an interaction into its graph representation. 

## Description

IN THIS SCENARIO, A MAN, CARL, HAS TO TAKE HIS MEDICATION PILLS BUT CANNOT FIND THEM.

THE AGENT FINDS THEM THROUGH OBJECT RECOGNITION AND COMMUNICATES THIS TO CARL. 

CARL THEN FIND THE PILLS

## Prerequisites 
1. Download [GraphDB](http://graphdb.ontotext.com/)
2. Launch it
3. Create a repository, you can use [this configuration](https://github.com/leolani/cltl-knowledgerepresentation/blob/main/src/cltl/brain/ontologies/BASIC-REPOSITORY-CONFIG-GRAPHDB.ttl)

In [1]:
# Imports
import json
from datetime import date, datetime
from pathlib import Path
from random import getrandbits

import requests
from cltl.brain.long_term_memory import LongTermMemory
from cltl.brain.utils.helper_functions import brain_response_to_json
from cltl.commons.discrete import UtteranceType
from tqdm import tqdm

In [2]:
# Define contextual features
context_id = getrandbits(8)
place_id = getrandbits(8)
location = requests.get("https://ipinfo.io").json()
start_date = date(2021, 3, 12)


In [10]:
# Data
carl_scenario = (
    {"context_id": context_id,
     "date": start_date,
     "place": "Carl's room",
     "place_id": place_id,
     "country": location['country'],
     "region": location['region'],
     "city": location['city']},
    [{  # CARL SAYS CANNOT SEE HIS PILLS
        "chat": 1,
        "turn": 1,
        "author": {"label": "carl", "type": ["person"], 'uri': "http://cltl.nl/leolani/friends/carl-1"},
        "utterance": "I need to take my pills, but I cannot find them.",
        "utterance_type": UtteranceType.STATEMENT,
        "position": "0-25",
        "subject": {"label": "carl", "type": ["person"], 'uri': "http://cltl.nl/leolani/world/carl-1"},
        "predicate": {"label": "see", "uri": "http://cltl.nl/leolani/n2mu/see"},
        "object": {"label": "pills", "type": ["object", "medicine"], 'uri': "http://cltl.nl/leolani/world/pills-1"},
        "perspective": {"certainty": 1, "polarity": -1, "sentiment": -1},
        "timestamp": datetime.combine(start_date, datetime.now().time()),
        "context_id": context_id
    }, {  # THE AGENT USES ITS CAMERA TO LOOK FOR THEM. SEES CARL
        "visual": 1,
        "detection": 3,
        "source": {"label": "front-camera", "type": ["sensor"], 'uri': "http://cltl.nl/leolani/inputs/front-camera"},
        "image": None,
        "utterance_type": UtteranceType.EXPERIENCE,
        "region": [752, 46, 1700, 716],
        "item": {'label': 'carl', 'type': ['person'], 'id': None, 'uri': "http://cltl.nl/leolani/world/carl-1"},
        'confidence': 0.94,
        "timestamp": datetime.combine(start_date, datetime.now().time()),
        "context_id": context_id
    }, {  # THE AGENT USES ITS CAMERA TO LOOK FOR THEM. SEES A CHAIR
        "visual": 1,
        "detection": 1,
        "source": {"label": "front-camera", "type": ["sensor"], 'uri': "http://cltl.nl/leolani/inputs/front-camera"},
        "image": None,
        "utterance_type": UtteranceType.EXPERIENCE,
        "region": [752, 46, 1148, 716],
        "item": {'label': 'chair', 'type': ['chair'], 'id': 1, 'uri': "http://cltl.nl/leolani/world/chair-1"},
        'confidence': 0.68,
        "timestamp": datetime.combine(start_date, datetime.now().time()),
        "context_id": context_id
    }, {  # THE AGENT USES ITS CAMERA TO LOOK FOR THEM. SEES A TABLE
        "visual": 1,
        "detection": 2,
        "source": {"label": "front-camera", "type": ["sensor"], 'uri': "http://cltl.nl/leolani/inputs/front-camera"},
        "image": None,
        "utterance_type": UtteranceType.EXPERIENCE,
        "region": [752, 86, 1148, 816],
        "item": {'label': 'table', 'type': ['table'], 'id': 1, 'uri': "http://cltl.nl/leolani/world/table-1"},
        'confidence': 0.68,
        "timestamp": datetime.combine(start_date, datetime.now().time()),
        "context_id": context_id
    }, {  # THE AGENT USES ITS CAMERA TO LOOK FOR THEM. SEES PILLS
        "visual": 1,
        "detection": 4,
        "source": {"label": "front-camera", "type": ["sensor"], 'uri': "http://cltl.nl/leolani/inputs/front-camera"},
        "image": None,
        "utterance_type": UtteranceType.EXPERIENCE,
        "region": [752, 46, 1148, 716],
        "item": {'label': 'pills', "type": ["object", "medicine"], 'id': 1,
                 'uri': "http://cltl.nl/leolani/world/pills-1"},
        'confidence': 0.92,
        "timestamp": datetime.combine(start_date, datetime.now().time()),
        "context_id": context_id
    }, {  # THE AGENT SAYS THE PILLS ARE UNDER THE TABLE
        "chat": 1,
        "turn": 2,
        "author": {"label": "leolani", "type": ["robot"], 'uri': "http://cltl.nl/leolani/friends/leolani"},
        "utterance": "They are under the table.",
        "utterance_type": UtteranceType.STATEMENT,
        "position": "0-25",
        "subject": {"label": "pills", "type": ["object", "medicine"], 'uri': "http://cltl.nl/leolani/world/pills-1"},
        "predicate": {"label": "located under", "uri": "http://cltl.nl/leolani/n2mu/located-under"},
        "object": {"label": "table", "type": ["object"], 'uri': "http://cltl.nl/leolani/world/table-1"},
        "perspective": {"certainty": 1, "polarity": 1, "sentiment": 0},
        "timestamp": datetime.combine(start_date, datetime.now().time()),
        "context_id": context_id
    }, {  # CARL SAYS HE SEES THE PILLS
        "chat": 1,
        "turn": 3,
        "author": {"label": "carl", "type": ["person"],
                   'uri': "http://cltl.nl/leolani/friends/carl-1"},
        "utterance": "Oh! Got it.",
        "utterance_type": UtteranceType.STATEMENT,
        "position": "0-25",
        "subject": {"label": "carl", "type": ["person"], 'uri': "http://cltl.nl/leolani/world/carl-1"},
        "predicate": {"label": "see", "uri": "http://cltl.nl/leolani/n2mu/see"},
        "object": {"label": "pills", "type": ["object", "medicine"], 'uri': "http://cltl.nl/leolani/world/pills-1"},
        "perspective": {"certainty": 1, "polarity": 1, "sentiment": 1},
        "timestamp": datetime.combine(start_date, datetime.now().time()),
        "context_id": context_id,
    }
    ]
)

In [23]:

def main(scenario):
    # Create folders
    scenario_filepath = Path('./data/carl/')
    graph_filepath = scenario_filepath / Path('graph/')
    graph_filepath.mkdir(parents=True, exist_ok=True)

    # Create brain connection
    brain = LongTermMemory(address="http://localhost:7200/repositories/sandbox",  # Location to save accumulated graph
                           log_dir=graph_filepath,  # Location to save step-wise graphs
                           clear_all=False)  # To start from an empty brain

    # Loop through capsules
    all_capsules = []
    all_responses = []
    for (context_capsule, content_capsules) in tqdm([scenario]):
        print(f"\n\n---------------------------------------------------------------\n")

        # Create context
        response = brain.capsule_context(context_capsule)

        # Add information to the brain
        for capsule in content_capsules:
            response = brain.capsule_statement(capsule, reason_types=True, create_label=True)
            print(f"\nTriple: {capsule['triple']}\n")
            # if capsule['utterance_type'] == UtteranceType.STATEMENT:
            #     response = brain.capsule_statement(capsule, reason_types=True, create_label=True)
            #     print(f"\nTriple: {capsule['triple']}\n")
            # if capsule['utterance_type'] == UtteranceType.EXPERIENCE:
            #     response = brain.capsule_experience(capsule, create_label=True)
            #     print(f"\nEntity: {capsule['entity']}\n")

            capsule_json = brain_response_to_json(capsule)
            all_capsules.append(capsule_json)
            response_json = brain_response_to_json(response)
            all_responses.append(response_json)

    # Save responses 
    f = open(scenario_filepath / "capsules.json", "w")
    json.dump(all_capsules, f)
    f = open(scenario_filepath / "responses.json", "w")
    json.dump(all_responses, f)

In [27]:
carl_scenario =  ({"context_id": context_id,
     "date": start_date,
     "place": "Carl's room",
     "place_id": place_id,
     "country": location['country'],
     "region": location['region'],
     "city": location['city']},
                  [
                      {'chat': '6e7ead30-123f-4918-9c48-f2d46201bda3',
                       'turn': 'f5e9614e-0ce4-4012-996b-7addda060a08', 
                       'author': {'label': 'Castor', 'type': ['person'], 'uri': 'http://cltl.nl/leolani/world/castor'}, 
                       'utterance': 'I like kiwis', 
                       'utterance_type': 'STATEMENT', 
                       'position': '0-12', 'subject': {'label': 'castor', 'type': [], 'uri': None}, 
                       'predicate': {'label': 'like', 'type': [], 'uri': None}, 'object': {'label': 'mellon', 'type': [], 'uri': None}, 
                       'context_id': '6e7ead30-123f-4918-9c48-f2d46201bda3', 'timestamp': 1762963498573, 'perspective': {'polarity': 1.0, 'certainty': 1.0, 'sentiment': 1.0}}
                  ]
                 )

In [28]:
main(carl_scenario)

2025-11-12 18:05:04 -     INFO -                                    cltl.brain.LongTermMemory - Booted
2025-11-12 18:05:04 -     INFO -                                  cltl.brain.ThoughtGenerator - Booted
2025-11-12 18:05:04 -     INFO -                                  cltl.brain.LocationReasoner - Booted
2025-11-12 18:05:04 -     INFO -                                      cltl.brain.TypeReasoner - Booted
2025-11-12 18:05:04 -     INFO -                                   cltl.brain.TrustCalculator - Booted
  0%|          | 0/1 [00:00<?, ?it/s]2025-11-12 18:05:04 -     INFO -                                    cltl.brain.LongTermMemory - Context: context99




---------------------------------------------------------------



2025-11-12 18:05:04 -     INFO -                                      cltl.brain.TypeReasoner - Reasoned type of castor to: agent
2025-11-12 18:05:07 -  WARNING -                                      cltl.brain.TypeReasoner - Failed to query Wikidata: HTTPSConnectionPool(host='query.wikidata.org', port=443): Read timed out. (read timeout=3)
2025-11-12 18:05:07 -     INFO -                                      cltl.brain.TypeReasoner - Reasoned type of mellon to: None
2025-11-12 18:05:07 -  WARNING -                                        cltl.brain.RdfBuilder - Unknown type: mellon
2025-11-12 18:05:07 -     INFO -                                    cltl.brain.LongTermMemory - Triple in statement: castor_like_mellon [agent_->_])
2025-11-12 18:05:07 -     INFO -                                  cltl.brain.ThoughtGenerator - Entity Novelty: existing subject - new object 
2025-11-12 18:05:08 -     INFO -                                  cltl.brain.ThoughtGenerator - Overlaps: 8 subject ove


Triple: castor_like_mellon [agent_->_])

